In [ ]:

#Description: Load product images and metadata, use OpenAI vision to generate compelling product listings automatically.

# Install: pip install datasets
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
import json
import os
from pathlib import Path
import io
import base64
from dotenv import load_dotenv
from openai import OpenAI

# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")


load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

OUTPUT_DIR = Path("output")
IMAGES_DIR = Path("product_images")
JSON_FILE = OUTPUT_DIR / "generated_product_listings.json"

OUTPUT_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)

print("Project setup complete!")


# Load Dataset


print("Loading product dataset...")

try:
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:20]")
    print(f"✓ Loaded {len(dataset)} products")

    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")

except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using fallback local product data instead...")

    products_data = [
        {
            "id": 1,
            "productDisplayName": "Wireless Headphones",
            "masterCategory": "Electronics",
            "subCategory": "Audio",
            "articleType": "Headphones",
            "baseColour": "Black",
            "season": "All",
            "usage": "Casual",
            "price": 79.99,
            "image": None
        },
        {
            "id": 2,
            "productDisplayName": "Men's Casual T-Shirt",
            "masterCategory": "Apparel",
            "subCategory": "Topwear",
            "articleType": "Tshirts",
            "baseColour": "Blue",
            "season": "Summer",
            "usage": "Casual",
            "price": 24.99,
            "image": None
        }
    ]

    products_df = pd.DataFrame(products_data)

print(f"\n✓ Dataset prepared!")
print(f"Total products: {len(products_df)}")


# Helper Functions


def pil_to_base64(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

def save_dataset_image(image_obj, file_path: Path):
    if isinstance(image_obj, Image.Image):
        image_obj.save(file_path)
        return True
    return False

def generate_listing_with_vision(product_info: dict, image: Image.Image | None):
    prompt = f"""
You are an expert e-commerce copywriter.

Create a compelling product listing from the following product information.

Product info:
- Name: {product_info.get('productDisplayName', 'Unknown Product')}
- Category: {product_info.get('masterCategory', 'Unknown')}
- Subcategory: {product_info.get('subCategory', 'Unknown')}
- Type: {product_info.get('articleType', 'Unknown')}
- Color: {product_info.get('baseColour', 'Unknown')}
- Season: {product_info.get('season', 'Unknown')}
- Usage: {product_info.get('usage', 'Unknown')}
- Price: {product_info.get('price', 'Not provided')}

Use the product image to improve the listing.

Return JSON only with this structure:
{{
  "title": "...",
  "short_description": "...",
  "full_description": "...",
  "key_features": ["...", "...", "..."],
  "target_audience": "...",
  "seo_keywords": ["...", "...", "..."]
}}
"""

    content = [{"type": "input_text", "text": prompt}]

    if image is not None:
        image_b64 = pil_to_base64(image)
        content.append({
            "type": "input_image",
            "image_url": f"data:image/jpeg;base64,{image_b64}"
        })

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "user",
                "content": content
            }
        ]
    )

    return response.output_text


# Product Listing Class


class ProductListing:
    def __init__(self, product_id, original_name, category, listing_data):
        self.product_id = product_id
        self.original_name = original_name
        self.category = category
        self.title = listing_data.get("title", "")
        self.short_description = listing_data.get("short_description", "")
        self.full_description = listing_data.get("full_description", "")
        self.key_features = listing_data.get("key_features", [])
        self.target_audience = listing_data.get("target_audience", "")
        self.seo_keywords = listing_data.get("seo_keywords", [])

    def to_dict(self):
        return self.__dict__


# Generate Listings


generated_listings = []

for idx, row in products_df.head(10).iterrows():
    print(f"\nProcessing product {idx + 1}...")

    product_info = row.to_dict()
    product_id = product_info.get("id", idx)

    image = None
    if "image" in product_info and product_info["image"] is not None:
        try:
            if isinstance(product_info["image"], Image.Image):
                image = product_info["image"]
            elif isinstance(product_info["image"], dict) and "bytes" in product_info["image"]:
                image = Image.open(io.BytesIO(product_info["image"]["bytes"]))
        except Exception as e:
            print(f"Could not read image for product {product_id}: {e}")

    if image is not None:
        image_path = IMAGES_DIR / f"product_{product_id}.jpg"
        image.save(image_path)

    try:
        result_text = generate_listing_with_vision(product_info, image)
        listing_json = json.loads(result_text)

        listing = ProductListing(
            product_id=product_id,
            original_name=product_info.get("productDisplayName", "Unknown"),
            category=product_info.get("masterCategory", "Unknown"),
            listing_data=listing_json
        )

        generated_listings.append(listing.to_dict())
        print(f"✓ Listing generated for product {product_id}")

    except Exception as e:
        print(f"⚠ Failed to generate listing for product {product_id}: {e}")


# Export JSON


with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(generated_listings, f, indent=4, ensure_ascii=False)

print(f"\n✓ Export complete!")
print(f"Saved JSON to: {JSON_FILE}")


# Validate JSON


with open(JSON_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("\nJSON Validation:")
print("Type:", type(data))
if data:
    print("First record:")
    print(json.dumps(data[0], indent=4))
else:
    print("No listings generated.")

Loading product dataset...


✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']
Project setup complete!
Loading product dataset...
✓ Loaded 20 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
Total products: 20

Processing product 1...
✓ Listing generated for product 15970

Processing product 2...
✓ Listing generated for product 39386

Processing product 3...
⚠ Failed to generate listing for product 59263: Expecting value: line 1 column 1 (char 0)

Processing product 4...
⚠ Failed to generate listing for product 21379: Expecting value: line 1 column 1 (char 0)

Processing product 5...
✓ Listing generated for product 53759

Processing product 6...
✓ Listing generated for product 1855

Processing product 7...
⚠ Failed to generate listing for product 30805: Expe